In [1]:
import os
import pathlib
import shutil

import numpy as np
import pennylane as qml
from catalyst.debug import get_compilation_stage

for f in os.listdir():
    if f.startswith("circuit"):
        shutil.rmtree(pathlib.Path(f))


pipelines = [
    (
        "QuantumCompilationStage",
        [
            "split-multiple-tapes",
            "builtin.module(apply-transform-sequence)",
            "inline-nested-module",
            "lower-mitigation",
            "adjoint-lowering",
        ],
    ),
    (
        "HLOLoweringStage",
        [
            "canonicalize",
            "func.func(chlo-legalize-to-stablehlo)",
            "func.func(stablehlo-legalize-control-flow)",
            "func.func(stablehlo-aggressive-simplification)",
            "stablehlo-legalize-to-linalg",
            "func.func(stablehlo-legalize-to-std)",
            "func.func(stablehlo-legalize-sort)",
            "stablehlo-convert-to-signless",
            "canonicalize",
            "scatter-lowering",
            "hlo-custom-call-lowering",
            "cse",
            "func.func(linalg-detensorize{aggressive-mode})",
            "detensorize-scf",
            "detensorize-function-boundary",
            "canonicalize",
            "symbol-dce",
        ],
    ),
    (
        "IonDecompositionStage",
        [
            "ions-decomposition",
            "merge-rotations",
            "prune-zero-rotations",
            "merge-rotations",
        ],
    ),
    (
        "GradientLoweringStage",
        [
            "annotate-invalid-gradient-functions",
            "lower-gradients",
        ],
    ),
    (
        "BufferizationStage",
        [
            "inline",
            "convert-tensor-to-linalg",
            "convert-elementwise-to-linalg",
            "gradient-preprocess",
            "one-shot-bufferize{bufferize-function-boundaries         allow-return-allocs-from-loops         function-boundary-type-conversion=identity-layout-map         unknown-type-conversion=identity-layout-map}",
            "canonicalize",
            "gradient-postprocess",
            "func.func(buffer-hoisting)",
            "func.func(buffer-loop-hoisting)",
            "func.func(buffer-deallocation)",
            "convert-arraylist-to-memref",
            "convert-bufferization-to-memref",
            "canonicalize",
            "cp-global-memref",
        ],
    ),
    (
        "MLIRToLLVMDialectConversion",
        [
            "expand-realloc",
            "convert-gradient-to-llvm",
            "memrefcpy-to-linalgcpy",
            "func.func(convert-linalg-to-loops)",
            "convert-scf-to-cf",
            "expand-strided-metadata",
            "lower-affine",
            "arith-expand",
            "convert-complex-to-standard",
            "convert-complex-to-llvm",
            "convert-math-to-llvm",
            "convert-math-to-libm",
            "convert-arith-to-llvm",
            "memref-to-llvm-tbaa",
            "finalize-memref-to-llvm{use-generic-functions}",
            "convert-index-to-llvm",
            "convert-catalyst-to-llvm",
            "convert-quantum-to-llvm",
            "emit-catalyst-py-interface",
            "canonicalize",
            "reconcile-unrealized-casts",
            "gep-inbounds",
            "register-inactive-callback",
        ],
    ),
]


device = qml.device("null.qubit", wires=26)

with open("steane.qasm", "r") as f:
    qasm_string = f.read()


@qml.qjit(keep_intermediate=True)
@qml.qnode(device)
def circuit():
    qml.from_qasm(qasm_string)()
    return qml.state()


@qml.qjit(keep_intermediate=True, pipelines=pipelines, verbose=True)
@qml.qnode(device)
def circuit_compiled():
    qml.from_qasm(qasm_string)()
    return qml.state()


print("{:=^100}".format("\033[1;31m Original circuit \033[0m"))
print(circuit.mlir)

print("{:=^100}".format("\033[1;32m Compiled circuit \033[0m"))
print(get_compilation_stage(circuit_compiled, stage="IonDecompositionStage"))


/home/user/projects/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane_qiskit/converter.py:585: UserWarning: pennylane_qiskit.converter: The Reset instruction is not supported by PennyLane, and has not been added to the template.
  warnings.warn(
/home/user/projects/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane_qiskit/converter.py:585: UserWarning: pennylane_qiskit.converter: The Reset instruction is not supported by PennyLane, and has not been added to the template.
  warnings.warn(


[LIB] Running compiler driver in /home/user/projects/oqd-catalyst/scripts/circuit_compiled
[SYSTEM] /home/user/projects/oqd-catalyst/frontend/catalyst/utils/../../../mlir/build/bin/catalyst -o /home/user/projects/oqd-catalyst/scripts/circuit_compiled/circuit_compiled.ll --module-name circuit_compiled --workspace /home/user/projects/oqd-catalyst/scripts/circuit_compiled -verify-each=false --catalyst-pipeline QuantumCompilationStage(split-multiple-tapes;builtin.module(apply-transform-sequence);inline-nested-module;lower-mitigation;adjoint-lowering),HLOLoweringStage(canonicalize;func.func(chlo-legalize-to-stablehlo);func.func(stablehlo-legalize-control-flow);func.func(stablehlo-aggressive-simplification);stablehlo-legalize-to-linalg;func.func(stablehlo-legalize-to-std);func.func(stablehlo-legalize-sort);stablehlo-convert-to-signless;canonicalize;scatter-lowering;hlo-custom-call-lowering;cse;func.func(linalg-detensorize{aggressive-mode});detensorize-scf;detensorize-function-boundary;canoni

In [2]:
print(qml.specs(circuit)())

/home/user/projects/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane_qiskit/converter.py:585: UserWarning: pennylane_qiskit.converter: The Reset instruction is not supported by PennyLane, and has not been added to the template.
  warnings.warn(


Device: null.qubit
Device wires: 26
Shots: Shots(total=None)
Level: device

Resource specifications:
  Total wire allocations: 26
  Total gates: 108
  Circuit depth: 24

  Gate types:
    Hadamard: 14
    S: 28
    CNOT: 24
    RY: 14
    RZ: 28

  Measurements:
    No measurements.


/home/user/projects/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane/resource/specs.py:123: UserWarning: Measurement resource tracking is not yet supported for qjit'd QNodes. The returned SpecsResources will have an empty measurements field.
  warnings.warn(


In [3]:
print(qml.specs(circuit_compiled)())

/home/user/projects/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane_qiskit/converter.py:585: UserWarning: pennylane_qiskit.converter: The Reset instruction is not supported by PennyLane, and has not been added to the template.
  warnings.warn(
[LIB] Running compiler driver in /home/user/projects/oqd-catalyst/scripts/circuit_compiled_1
[SYSTEM] /home/user/projects/oqd-catalyst/frontend/catalyst/utils/../../../mlir/build/bin/catalyst -o /home/user/projects/oqd-catalyst/scripts/circuit_compiled_1/circuit_compiled.ll --module-name circuit_compiled --workspace /home/user/projects/oqd-catalyst/scripts/circuit_compiled_1 -verify-each=false --catalyst-pipeline QuantumCompilationStage(split-multiple-tapes;builtin.module(apply-transform-sequence);inline-nested-module;lower-mitigation;adjoint-lowering),HLOLoweringStage(canonicalize;func.func(chlo-legalize-to-stablehlo);func.func(stablehlo-legalize-control-flow);func.func(stablehlo-aggressive-simplification);stablehlo-legalize-to-linalg;

Device: null.qubit
Device wires: 26
Shots: Shots(total=None)
Level: device

Resource specifications:
  Total wire allocations: 26
  Total gates: 233
  Circuit depth: 49

  Gate types:
    MS: 24
    RY: 108
    RX: 101

  Measurements:
    No measurements.


/home/user/projects/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane/resource/specs.py:123: UserWarning: Measurement resource tracking is not yet supported for qjit'd QNodes. The returned SpecsResources will have an empty measurements field.
  warnings.warn(


In [4]:
print(qml.draw(circuit)())

0: ─╭||──RZ(1.57)──RY(0.00)──RZ(1.57)─╭||────╭||───────╭||────╭||────╭||────╭||─╭●─╭||──────╭|| ···
1: ─├||──RZ(1.57)──RY(0.00)──RZ(1.57)─├||────├||───────├||─╭●─├||─╭●─├||────├||─│──├||──────├|| ···
2: ─├||───────────────────────────────├||─╭X─├||────╭X─├||─│──├||─╰X─├||─╭X─├||─│──├||──┤↗├─├|| ···
3: ─├||──RZ(1.57)──RY(0.00)──RZ(1.57)─├||─╰●─├||────│──├||─│──├||────├||─│──├||─│──├||──────├|| ···
4: ─├||───────────────────────────────├||─╭X─├||─╭X─│──├||─╰X─├||────├||─│──├||─╰X─├||──┤↗├─├|| ···
5: ─├||──RZ(1.57)──RY(0.00)──RZ(1.57)─├||─╰●─├||─│──│──├||────├||────├||─╰●─├||─╭●─├||──────├|| ···
6: ─├||──RZ(1.57)──RY(0.00)──RZ(1.57)─├||────├||─│──╰●─├||─╭●─├||────├||────├||─│──├||──────├|| ···
7: ─├||──RZ(1.57)──RY(0.00)──RZ(1.57)─├||────├||─╰●────├||─│──├||─╭●─├||────├||─│──├||──────├|| ···
8: ─├||───────────────────────────────├||────├||───────├||─╰X─├||─╰X─├||─╭X─├||─╰X─├||──┤↗├─├|| ···
9: ─╰||──RZ(1.57)──RY(0.00)──RZ(1.57)─╰||────╰||───────╰||────╰||────╰||─╰●─╰||────╰||──────╰|| ···


/home/user/projects/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane_qiskit/converter.py:585: UserWarning: pennylane_qiskit.converter: The Reset instruction is not supported by PennyLane, and has not been added to the template.
  warnings.warn(
